In [ ]:
from google.colab import files
uploaded = files.upload()

Saving hindi to english dataset.zip to hindi to english dataset (1).zip


In [ ]:
import os

print(os.listdir())

['.config', 'hindi to english dataset.zip', 'hindi to english dataset (1).zip', 'data', 'sample_data']


In [ ]:
import zipfile

zip_ref = zipfile.ZipFile('hindi to english dataset.zip', 'r')
zip_ref.extractall('/content/data')
zip_ref.close()

print("Dataset extracted successfully!")


Dataset extracted successfully!


In [ ]:
import os

print(os.listdir('/content/data'))

['__MACOSX', 'hindi to english dataset']


In [ ]:
import os

print(os.listdir('/content/data/hindi to english dataset'))


['opus.en-hi-dev.en', 'opus.en-hi-dev.hi', 'opus.en-hi-test.hi', 'opus.en-hi-train.hi', 'opus.en-hi-test.en', 'opus.en-hi-train.en']


In [ ]:
train_en_path = "/content/data/hindi to english dataset/opus.en-hi-train.en"
train_hi_path = "/content/data/hindi to english dataset/opus.en-hi-train.hi"

with open(train_en_path, "r", encoding="utf-8") as f:
    english_sentences = f.readlines()

with open(train_hi_path, "r", encoding="utf-8") as f:
    hindi_sentences = f.readlines()

print("Total English sentences:", len(english_sentences))
print("Total Hindi sentences:", len(hindi_sentences))

print("\nSample English:")
print(english_sentences[0])

print("\nSample Hindi:")
print(hindi_sentences[0])

Total English sentences: 534319
Total Hindi sentences: 534319

Sample English:
Other, Private Use


Sample Hindi:
अन्य, निज़ी उपयोग



In [ ]:
test_sentence = "How are you"

tokens = test_sentence.lower().strip().split()

print(tokens)

['how', 'are', 'you']


In [ ]:
english_sentences = english_sentences[:10000]
hindi_sentences = hindi_sentences[:10000]

print(len(english_sentences))

10000


In [ ]:
def tokenize(sentence):
    return sentence.lower().strip().split()

print(tokenize(english_sentences[0]))
print(tokenize(hindi_sentences[0]))

['other,', 'private', 'use']
['अन्य,', 'निज़ी', 'उपयोग']


In [ ]:
from collections import Counter

english_counter = Counter()
hindi_counter = Counter()

for sentence in english_sentences:
    english_counter.update(tokenize(sentence))

for sentence in hindi_sentences:
    hindi_counter.update(tokenize(sentence))

print("English vocabulary size:", len(english_counter))
print("Hindi vocabulary size:", len(hindi_counter))

English vocabulary size: 16338
Hindi vocabulary size: 16323


In [ ]:
english_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "<UNK>": 3
}

hindi_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "<UNK>": 3
}

for word in english_counter:
    english_vocab[word] = len(english_vocab)

for word in hindi_counter:
    hindi_vocab[word] = len(hindi_vocab)

print("English vocab size:", len(english_vocab))
print("Hindi vocab size:", len(hindi_vocab))

print("\nExample English word mapping:")
print(list(english_vocab.items())[:10])

print("\nExample Hindi word mapping:")
print(list(hindi_vocab.items())[:10])

English vocab size: 16342
Hindi vocab size: 16327

Example English word mapping:
[('<PAD>', 0), ('<SOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('other,', 4), ('private', 5), ('use', 6), ('[screaming]', 7), ('spouse', 8), ('i', 9)]

Example Hindi word mapping:
[('<PAD>', 0), ('<SOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('अन्य,', 4), ('निज़ी', 5), ('उपयोग', 6), ('ऊबड़', 7), ('.', 8), ('जीवनसाथी', 9)]


In [ ]:
def sentence_to_indices(sentence, vocab):
    tokens = tokenize(sentence)

    indices = [vocab["<SOS>"]]

    for token in tokens:
        indices.append(vocab.get(token, vocab["<UNK>"]))

    indices.append(vocab["<EOS>"])

    return indices


sample_english = english_sentences[0]
sample_hindi = hindi_sentences[0]

english_indices = sentence_to_indices(sample_english, english_vocab)
hindi_indices = sentence_to_indices(sample_hindi, hindi_vocab)

print("English sentence:")
print(sample_english)

print("\nEnglish indices:")
print(english_indices)

print("\nHindi sentence:")
print(sample_hindi)

print("\nHindi indices:")
print(hindi_indices)

English sentence:
Other, Private Use


English indices:
[1, 4, 5, 6, 2]

Hindi sentence:
अन्य, निज़ी उपयोग


Hindi indices:
[1, 4, 5, 6, 2]


In [ ]:
MAX_LENGTH = 20

def pad_sequence(sequence, max_length):

    if len(sequence) < max_length:
        sequence = sequence + [0] * (max_length - len(sequence))

    else:
        sequence = sequence[:max_length]

    return sequence


sample_padded = pad_sequence(english_indices, MAX_LENGTH)

print("Original sequence:")
print(english_indices)

print("\nPadded sequence:")
print(sample_padded)

print("\nLength:", len(sample_padded))

Original sequence:
[1, 4, 5, 6, 2]

Padded sequence:
[1, 4, 5, 6, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Length: 20


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class TranslationDataset(Dataset):

    def __init__(self, english_sentences, hindi_sentences,
                 english_vocab, hindi_vocab, max_length):

        self.english_sentences = english_sentences
        self.hindi_sentences = hindi_sentences
        self.english_vocab = english_vocab
        self.hindi_vocab = hindi_vocab
        self.max_length = max_length


    def __len__(self):
        return len(self.english_sentences)


    def __getitem__(self, idx):

        english_seq = sentence_to_indices(
            self.english_sentences[idx],
            self.english_vocab
        )

        hindi_seq = sentence_to_indices(
            self.hindi_sentences[idx],
            self.hindi_vocab
        )

        english_seq = pad_sequence(english_seq, self.max_length)
        hindi_seq = pad_sequence(hindi_seq, self.max_length)

        return (
            torch.tensor(english_seq),
            torch.tensor(hindi_seq)
        )


dataset = TranslationDataset(
    english_sentences,
    hindi_sentences,
    english_vocab,
    hindi_vocab,
    MAX_LENGTH
)

dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

print("Dataset size:", len(dataset))

Dataset size: 10000


In [ ]:
import torch.nn as nn


class TransformerModel(nn.Module):

    def __init__(
        self,
        english_vocab_size,
        hindi_vocab_size,
        embed_size=256,
        num_heads=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        forward_expansion=512,
        dropout=0.1,
        max_length=20
    ):

        super(TransformerModel, self).__init__()

        self.english_embedding = nn.Embedding(
            english_vocab_size,
            embed_size
        )

        self.hindi_embedding = nn.Embedding(
            hindi_vocab_size,
            embed_size
        )

        self.position_embedding = nn.Embedding(
            max_length,
            embed_size
        )

        self.transformer = nn.Transformer(
            d_model=embed_size,
            nhead=num_heads,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=forward_expansion,
            dropout=dropout
        )

        self.fc_out = nn.Linear(embed_size, hindi_vocab_size)

        self.dropout = nn.Dropout(dropout)


    def forward(self, source, target):

        batch_size, source_length = source.shape
        batch_size, target_length = target.shape

        source_positions = (
            torch.arange(0, source_length)
            .unsqueeze(0)
            .expand(batch_size, source_length)
            .to(source.device)
        )

        target_positions = (
            torch.arange(0, target_length)
            .unsqueeze(0)
            .expand(batch_size, target_length)
            .to(target.device)
        )

        embed_source = self.dropout(
            self.english_embedding(source)
            + self.position_embedding(source_positions)
        )

        embed_target = self.dropout(
            self.hindi_embedding(target)
            + self.position_embedding(target_positions)
        )

        embed_source = embed_source.permute(1, 0, 2)
        embed_target = embed_target.permute(1, 0, 2)

        output = self.transformer(
            embed_source,
            embed_target
        )

        output = self.fc_out(output)

        return output

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerModel(
    len(english_vocab),
    len(hindi_vocab)
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)
print("\nUsing device:", device)

TransformerModel(
  (english_embedding): Embedding(16342, 256)
  (hindi_embedding): Embedding(16327, 256)
  (position_embedding): Embedding(20, 256)
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): T

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


In [ ]:
num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for batch_idx, (source, target) in enumerate(dataloader):

        source = source.to(device)
        target = target.to(device)

        target_input = target[:, :-1]
        target_output = target[:, 1:]

        output = model(source, target_input)

        output = output.reshape(-1, output.shape[2])
        target_output = target_output.reshape(-1)

        optimizer.zero_grad()

        loss = criterion(output, target_output)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")


Epoch 1/5, Loss: 7.4068
Epoch 2/5, Loss: 7.1271
Epoch 3/5, Loss: 7.0963
Epoch 4/5, Loss: 7.0877
Epoch 5/5, Loss: 7.0865


In [ ]:
index_to_hindi = {index: word for word, index in hindi_vocab.items()}


def translate(sentence, max_length=20):

    model.eval()

    tokens = tokenize(sentence)

    indices = [english_vocab["<SOS>"]]

    for token in tokens:
        indices.append(
            english_vocab.get(token, english_vocab["<UNK>"])
        )

    indices.append(english_vocab["<EOS>"])

    source = torch.tensor(indices).unsqueeze(0).to(device)

    target_indices = [hindi_vocab["<SOS>"]]

    for _ in range(max_length):

        target_tensor = torch.tensor(target_indices).unsqueeze(0).to(device)

        with torch.no_grad():

            output = model(source, target_tensor)

        next_token = output.argmax(2)[:, -1].item()

        target_indices.append(next_token)

        if next_token == hindi_vocab["<EOS>"]:
            break

    translated_tokens = []

    for index in target_indices:

        word = index_to_hindi.get(index, "<UNK>")

        if word not in ["<SOS>", "<EOS>", "<PAD>"]:
            translated_tokens.append(word)

    return " ".join(translated_tokens)

In [ ]:
print(translate("Other Private Use"))

In [ ]:
print(translate("Private"))

In [ ]:
print(translate("How are you"))

In [ ]:
num_epochs = 5
num_epochs = 15
num_epochs = 20

In [ ]:
num_epochs = 20

In [ ]:
print(translate("How are you"))

In [ ]:
num_epochs = 5

In [ ]:
num_epochs = 20

In [ ]:
num_epochs = 30

In [ ]:
english_sentences = english_sentences[:50000]
hindi_sentences = hindi_sentences[:50000]